In [1]:
import pandas as pd
import importlib
import process_utils as pu
importlib.reload(pu)
from process_utils import MultipleRun

# Statistical Significance Testing

In [2]:
AL_SETUP = {
    'HAR': (3,300),
    'UWAVE' : (3,25),
    'GRABMYO': (6,20),
    'WISDM': (5,40)
}

AL_METHODS = ['Random', 'TypiClust', 'CoreSet'] # 'TypiCore', 'Kmeans-ppp', 'CoreSetProb'
CIL_METHODS = ['ER', 'ASER']

In [3]:
multiple_t_tests = []

for dataset in AL_SETUP.keys():
    al_budget = AL_SETUP[dataset][0]
    al_total = AL_SETUP[dataset][1]
    print(f"Processing dataset: {dataset} with AL budget: {al_budget} and total AL steps: {al_total}")


    multiple_runs = []
    for cil in CIL_METHODS:
        for al_method in AL_METHODS:
            multiple_runs.append(MultipleRun(dataset=dataset,
                                            CIL_method=cil,
                                            AL_method=al_method,
                                            al_budget=al_budget,
                                            al_total=al_total))
            
    multiple_t_tests.append(pu.perform_paired_t_test(multiple_runs))


Processing dataset: HAR with AL budget: 3 and total AL steps: 300
Processing dataset: UWAVE with AL budget: 3 and total AL steps: 25
Processing dataset: GRABMYO with AL budget: 6 and total AL steps: 20
Processing dataset: WISDM with AL budget: 5 and total AL steps: 40


In [4]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.3f}'.format)

In [5]:
multiple_t_tests[0]

,Dataset,CIL,AL_Method,Metric,Baseline_Mean,Method_Mean,Diff,t_stat,p_better,sig_better,p_worse,sig_worse
0,har,ER,TypiClust,A_T,67.836,75.262,7.426,2.485,0.034,*,0.966,
1,har,ER,TypiClust,A_curr,91.947,95.239,3.293,3.216,0.016,*,0.984,
2,har,ER,TypiClust,F_T,36.302,29.966,-6.336,-1.555,0.097,,0.903,
3,har,ER,CoreSet,A_T,67.836,63.193,-4.643,-0.976,0.808,,0.192,
4,har,ER,CoreSet,A_curr,91.947,94.966,3.019,2.972,0.021,*,0.979,
5,har,ER,CoreSet,F_T,36.302,47.659,11.357,1.718,0.920,,0.080,
6,har,ASER,TypiClust,A_T,63.005,62.515,-0.489,-0.108,0.541,,0.459,
7,har,ASER,TypiClust,A_curr,93.834,93.862,0.028,0.016,0.494,,0.506,
8,har,ASER,TypiClust,F_T,46.244,47.020,0.776,0.096,0.536,,0.464,
9,har,ASER,CoreSet,A_T,63.005,63.034,0.029,0.006,0.498,,0.502,


In [6]:
sig = pd.concat(multiple_t_tests)


In [7]:
sigsel = sig[(sig['sig_better'] != "") | (sig['sig_worse'] != "")]

In [8]:
sigsel = sigsel.reset_index(drop=True)
sigsel

,Dataset,CIL,AL_Method,Metric,Baseline_Mean,Method_Mean,Diff,t_stat,p_better,sig_better,p_worse,sig_worse
0,har,ER,TypiClust,A_T,67.836,75.262,7.426,2.485,0.034,*,0.966,
1,har,ER,TypiClust,A_curr,91.947,95.239,3.293,3.216,0.016,*,0.984,
2,har,ER,CoreSet,A_curr,91.947,94.966,3.019,2.972,0.021,*,0.979,
3,grabmyo,ER,TypiClust,A_T,51.246,54.892,3.646,2.584,0.031,*,0.969,
4,grabmyo,ER,CoreSet,A_T,51.246,44.090,-7.156,-4.071,0.992,,0.008,**
5,grabmyo,ER,CoreSet,A_curr,67.662,86.574,18.913,2.653,0.028,*,0.972,
6,grabmyo,ER,CoreSet,F_T,23.988,53.106,29.117,4.900,0.996,,0.004,**
7,grabmyo,ASER,CoreSet,A_T,49.786,41.191,-8.595,-3.291,0.985,,0.015,*
8,grabmyo,ASER,CoreSet,A_curr,73.379,84.178,10.800,2.865,0.023,*,0.977,
9,grabmyo,ASER,CoreSet,F_T,31.382,53.847,22.465,3.838,0.991,,0.009,**


# Current Accuracy


---
## Investigating t-statistic

Let's inspect the raw `F_T` (Forgetting) scores for the two experiments you highlighted. This will help visualize why a smaller difference can have a higher significance. We will look at the scores for each individual run.
